# Value Definition

$$ 
V^{\pi}(s) = \mathbb{E}_{\pi} [G_t \mid S_t = s]
$$

# Bellman Expectation Equation (Policy Evaluation)

$$
V^{\pi}(s) =
\sum_a \pi(a|s)
\sum_{s',r}
p(s',r|s,a)
\left[
r + \gamma V^{\pi}(s')
\right]
$$

### Iterative Update Rule

$$
V(s) \leftarrow
\sum_a \pi(a|s)
\sum_{s',r}
p(s',r|s,a)
\left[
r + \gamma V(s')
\right]
$$

In [1]:
import numpy as np

In [2]:
# states
states = [0, 1, 2]

# actions
actions = [0, 1]   # 0=left, 1=right

gamma = 0.9
theta = 1e-4

# policy π(a|s)  (uniform policy)
policy = {
    0: [0.5, 0.5],
    1: [0.5, 0.5],
    2: [0.5, 0.5]
}

In [3]:
# transition model p(s', r | s, a)
# format: P[s][a] = list of (prob, next_state, reward)
P = {
    0: {
        0: [(1.0, 0, 0)],      # left -> stay
        1: [(1.0, 1, 0)]       # right -> state 1
    },
    1: {
        0: [(1.0, 0, 0)],
        1: [(1.0, 2, 1)]       # reward when reaching state 2
    },
    2: {
        0: [(1.0, 2, 0)],
        1: [(1.0, 2, 0)]       # terminal-like
    }
}

In [4]:
# initialize V(s)=0
V = np.zeros(len(states))

In [5]:
def iterative_policy_evaluation():
    global V
    
    while True:
        delta = 0
        
        for s in states:
            
            v = V[s]
            new_v = 0
            
            for a, action_prob in enumerate(policy[s]):
                
                for prob, next_state, reward in P[s][a]:
                    
                    new_v += action_prob * prob * (
                        reward + gamma * V[next_state]
                    )
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        if delta < theta:
            break
    
    return V

In [6]:
V_result = iterative_policy_evaluation()

print("State Values:")
for s in states:
    print(f"V({s}) = {V_result[s]:.4f}")

State Values:
V(0) = 0.6474
V(1) = 0.7913
V(2) = 0.0000


# Policy Improvement Equation

$$
\pi'(s) =
\arg\max_a
\sum_{s',r}
p(s',r|s,a)
\left[
r + \gamma V(s')
\right]
$$

## Action Value (Q function)
$$
Q(s,a) =
\sum_{s',r}
p(s',r|s,a)
\left[
r + \gamma V(s')
\right]
$$

## Greedy Policy
$$
\pi'(s) = \arg\max_a Q(s,a)
$$

1. Policy Evaluation
$$
V^{\pi}(s)
$$

2. Policy Improvement
$$
\pi'(s) = \arg\max_a Q(s,a)
$$

In [7]:
states = [0,1,2]
actions = [0,1]  # 0=left, 1=right

gamma = 0.9
theta = 1e-4

In [8]:
# transition model
# P[s][a] = (prob, next_state, reward)
P = {
    0:{
        0:[(1.0,0,0)],
        1:[(1.0,1,0)]
    },
    1:{
        0:[(1.0,0,0)],
        1:[(1.0,2,1)]
    },
    2:{
        0:[(1.0,2,0)],
        1:[(1.0,2,0)]
    }
}

In [9]:
# initialize
V = np.zeros(len(states))
policy = np.random.choice(actions, size=len(states))

In [10]:
def policy_evaluation(policy, V):

    while True:
        delta = 0

        for s in states:

            v = V[s]
            a = policy[s]

            new_v = 0

            for prob, next_state, reward in P[s][a]:
                new_v += prob * (reward + gamma * V[next_state])

            V[s] = new_v

            delta = max(delta, abs(v - V[s]))

        if delta < theta:
            break

    return V


def policy_improvement(V, policy):

    policy_stable = True

    for s in states:

        old_action = policy[s]

        action_values = []

        for a in actions:

            q = 0

            for prob, next_state, reward in P[s][a]:
                q += prob * (reward + gamma * V[next_state])

            action_values.append(q)

        best_action = np.argmax(action_values)

        policy[s] = best_action

        if old_action != best_action:
            policy_stable = False

    return policy, policy_stable


def policy_iteration():

    global V, policy

    while True:

        V = policy_evaluation(policy, V)

        policy, stable = policy_improvement(V, policy)

        if stable:
            break

    return policy, V

In [11]:
optimal_policy, optimal_V = policy_iteration()

print("Optimal Policy:", optimal_policy)
print("Value Function:", optimal_V)

Optimal Policy: [1 1 0]
Value Function: [0.9 1.  0. ]
